In [2]:
# --- Read parameters from the URL (synced from FrankenMSA web tool) ---
from urllib.parse import parse_qs
from google.colab import output

_qs = output.eval_js("location.search") or ""
params = {k: v[0] for k, v in parse_qs(_qs[1:]).items()}  

def _get_float(key, default):
    try:
        return float(params.get(key, default))
    except Exception:
        return float(default)

def _get_int(key, default):
    try:
        return int(params.get(key, default))
    except Exception:
        return int(default)

def _norm_csv(key):
    s = (params.get(key, "") or "").replace(" ", "").upper()
    return s

sampling_temp = _get_float("temp", 1.0)
num_seqs      = _get_int("num", 128)
pdb_code      = (_norm_csv("pdb") or "")
homomer       = (params.get("homomer", "1") == "1")

designed_chain_csv = _norm_csv("design")
fixed_chain_csv    = _norm_csv("fixed")

designed_chain_list = [c for c in designed_chain_csv.split(",") if c] if designed_chain_csv else []
fixed_chain_list    = [c for c in fixed_chain_csv.split(",") if c]    if fixed_chain_csv else []

print({
    "sampling_temp": sampling_temp,
    "num_seqs": num_seqs,
    "pdb_code": pdb_code,
    "homomer": homomer,
    "design": designed_chain_list,
    "fixed": fixed_chain_list,
})

ModuleNotFoundError: No module named 'google'

In [ ]:
import os, sys, subprocess, json, re

# Clone ProteinMPNN repo if not already present
if not os.path.isdir("ProteinMPNN"):
    subprocess.run(["git", "clone", "-q", "https://github.com/dauparas/ProteinMPNN.git"], check=True)
sys.path.append("/content/ProteinMPNN")

# Install dependencies quietly
!pip install -q biopython==1.83 einops==0.7.0

print("✅ ProteinMPNN repo ready and dependencies installed.")

In [ ]:
#  Read URL-like parameters (for syncing Web UI -> Colab)
# -----------------------------------------------------------------
# How to use:
#   1) Set QUERY to something like:
#        "?temp=0.2&num=256&model=v_48_020&soluble=0&ca=0&design=A,B&fixed=&pdb=1091&homomer=1"
#   2) Run this cell. It will set/override the notebook variables:
#        sampling_temp, num_seqs, model_name, use_soluble_model, ca_only,
#        designed_chains, fixed_chains, pdb_code
#   3) The later cells (get_pdb / run ProteinMPNN) will pick these up.

from urllib.parse import parse_qs
import re

# <<< Put your query string here (the web app can generate this) >>>
QUERY = ""  # e.g. "?temp=0.2&num=128&model=v_48_020&soluble=0&ca=0&design=A&fixed=&pdb=1091&homomer=1"

def _to_bool(x):
    # Accept "1/0/true/false/yes/no" (case-insensitive). Default False on empty.
    if x is None:
        return False
    s = str(x).strip().lower()
    return s in ("1", "true", "yes", "y", "t")

def _to_int(x, default):
    try:
        return int(x)
    except Exception:
        return default

def _nonempty(s):
    return s is not None and str(s).strip() != ""

def parse_query(query: str):
    if not query:
        return {}
    if query.startswith("?"):
        query = query[1:]
    kv = {k: v[0] if isinstance(v, list) and v else v for k, v in parse_qs(query).items()}
    return kv

params = parse_query(QUERY)

# --- Map query keys -> notebook variables with sane defaults ---
# Temperature: allow "0.1" or multiple like "0.1 0.2 0.3"
sampling_temp = params.get("temp") or globals().get("sampling_temp", "0.1")
# Number of sequences
num_seqs = _to_int(params.get("num"), globals().get("num_seqs", 128))
# Model name
model_name = params.get("model") or globals().get("model_name", "v_48_020")
# Use soluble weights / CA-only
use_soluble_model = _to_bool(params.get("soluble")) if "soluble" in params else globals().get("use_soluble_model", False)
ca_only = _to_bool(params.get("ca")) if "ca" in params else globals().get("ca_only", False)

# Chains to design / fix: comma- or space-separated, letters only
_design = params.get("design", "")
_fixed  = params.get("fixed", "")
designed_chains = [c for c in re.split(r"[,\s]+", _design.strip()) if re.fullmatch(r"[A-Za-z]", c)] if _nonempty(_design) else []
fixed_chains    = [c for c in re.split(r"[,\s]+", _fixed.strip())  if re.fullmatch(r"[A-Za-z]", c)] if _nonempty(_fixed)  else []

# Optional PDB code (used by get_pdb cell)
pdb_code = params.get("pdb") or globals().get("pdb_code", "")

# Optional homomer flag (if you want to tie symmetric chains in your own logic)
homomer = _to_bool(params.get("homomer")) if "homomer" in params else globals().get("homomer", True)

print("✅ Parsed parameters:")
print(f"  sampling_temp      = {sampling_temp}")
print(f"  num_seqs           = {num_seqs}")
print(f"  model_name         = {model_name}")
print(f"  use_soluble_model  = {use_soluble_model}")
print(f"  ca_only            = {ca_only}")
print(f"  designed_chains    = {designed_chains or 'ALL'}")
print(f"  fixed_chains       = {fixed_chains or 'NONE'}")
print(f"  pdb_code           = {pdb_code or '(upload prompted)'}")
print(f"  homomer            = {homomer}")

In [ ]:
import re
from google.colab import files

# SUpload or download a PDB file
# ---------------------------------------------------------------
# This function either:
# (1) lets the user upload a local PDB file, or
# (2) automatically downloads a PDB file from the RCSB database by its PDB code.
def get_pdb(pdb_code=""):
    if pdb_code is None or pdb_code.strip() == "":
        # Option 1: User uploads a PDB file manually
        print("📤 Please upload a PDB file...")
        upload_dict = files.upload()
        pdb_filename = list(upload_dict.keys())[0]
        print(f"✅ Uploaded file: {pdb_filename}")
        return pdb_filename
    else:
        # Option 2: Download PDB by ID (e.g., 1ABC)
        pdb_filename = f"{pdb_code}.pdb"
        os.system(f"wget -q https://files.rcsb.org/view/{pdb_filename}")
        print(f"✅ Downloaded PDB: {pdb_filename}")
        return pdb_filename

# Example usage:
# pdb_path = get_pdb("1ABC")   # Download from RCSB
# pdb_path = get_pdb()         # Upload manually

In [ ]:
#Configure model & run ProteinMPNN
# ---------------------------------------------------------------
# This cell:
# 1) Sets model/params (temperature, number of sequences, model weights)
# 2) Calls the official ProteinMPNN CLI (protein_mpnn_run.py)
# 3) Collects the generated sequences into a single FASTA
# 4) (Optional) Writes a very simple A3M file from the FASTA

import os, sys, glob, json, subprocess, textwrap
from pathlib import Path

# ---- Inputs from previous cells / UI (edit as needed) ----
pdb_path = locals().get("pdb_path", None) or "tmp.pdb"  # filled by get_pdb()
num_seqs = 128                   # how many sequences to generate
sampling_temp = "0.1"            # e.g. "0.1", "0.2 0.3"
model_name = "v_48_020"          # v_48_002, v_48_010, v_48_020, v_48_030
use_soluble_model = False        # True to use weights trained on soluble proteins
ca_only = False                  # True for CA-only models
out_dir = "/content/ProteinMPNN/outputs_run"  # output folder

# Optional chain control (simple default: design all chains)
# If you want to design only chain A for example, set designed_chains = ["A"]
designed_chains = []  # [] means "design all"
fixed_chains = []     # chains to keep fixed

# ---- Resolve weights path based on flags ----
root = "/content/ProteinMPNN"
if ca_only:
    weights_root = f"{root}/ca_model_weights"
elif use_soluble_model:
    weights_root = f"{root}/soluble_model_weights"
else:
    weights_root = f"{root}/vanilla_model_weights"

assert os.path.isdir(weights_root), f"Model weights folder not found: {weights_root}"

os.makedirs(out_dir, exist_ok=True)

# ---- Optionally create a chain design spec via JSONL (only if user specified) ----
# protein_mpnn_run.py accepts --chain_id_jsonl for per-PDB design/fixed control.
chain_jsonl = None
if designed_chains or fixed_chains:
    chain_jsonl = os.path.join(out_dir, "chain_id.jsonl")
    name = Path(pdb_path).stem
    with open(chain_jsonl, "w") as f:
        obj = {
            "name": name,
            "design_chain_list": designed_chains,
            "fixed_chain_list": fixed_chains,
        }
        f.write(json.dumps(obj) + "\n")

# ---- Build the CLI command ----
cmd = [
    sys.executable, f"{root}/protein_mpnn_run.py",
    "--pdb_path", pdb_path,
    "--out_folder", out_dir,
    "--model_name", model_name,
    "--path_to_model_weights", weights_root,
    "--num_seq_per_target", str(num_seqs),
    "--sampling_temp", sampling_temp,
]
if use_soluble_model:
    cmd.append("--use_soluble_model")
if ca_only:
    cmd.append("--ca_only")
if chain_jsonl:
    cmd.extend(["--chain_id_jsonl", chain_jsonl])

print("🔧 Running ProteinMPNN with command:\n", " ".join(cmd))

# ---- Execute ----
proc = subprocess.run(cmd, capture_output=True, text=True)
print("=== STDOUT ===")
print(proc.stdout)
print("=== STDERR ===")
print(proc.stderr)
if proc.returncode != 0:
    raise RuntimeError("ProteinMPNN run failed.")

# ---- Collect outputs into a single FASTA ----
# ProteinMPNN writes per-target sequence files into out_dir; we merge them.
fasta_out = os.path.join(out_dir, f"{Path(pdb_path).stem}_proteinmpnn.fasta")
with open(fasta_out, "w") as fout:
    count = 0
    for fn in sorted(glob.glob(os.path.join(out_dir, "**", "*.fa*"), recursive=True)):
        with open(fn) as fin:
            for line in fin:
                if line.startswith(">"):
                    count += 1
                    fout.write(f">sample_{count}\n")
                else:
                    fout.write(line.strip() + "\n")

print(f"✅ Merged FASTA written to: {fasta_out}")

# ---- (Optional) Write a minimal A3M (FASTA content, no gaps; FrankenMSA can still ingest) ----
a3m_out = os.path.join(out_dir, f"{Path(pdb_path).stem}_proteinmpnn.a3m")
with open(fasta_out) as fin, open(a3m_out, "w") as fout:
    i = 0
    for line in fin:
        if line.startswith(">"):
            i += 1
            fout.write(f">sample_{i}\n")
        else:
            # A very naive FASTA->A3M (no insertions); good enough as pseudo-MSA rows
            fout.write(line.strip() + "\n")

print(f"✅ Minimal A3M written to: {a3m_out}")

# If you want Colab to offer download buttons:
try:
    from google.colab import files as colab_files
    print("⬇️  Use the Colab download dialog to save the files locally:")
    print(fasta_out, a3m_out)
    # Uncomment to auto-open download dialogs:
    # colab_files.download(fasta_out)
    # colab_files.download(a3m_out)
except Exception:
    pass